<a href="https://colab.research.google.com/github/kavya-83175/24B11DS151_DriveReady_OPC_2026/blob/main/FinMateAI_Day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q langchain langchain-groq gspread

import os
from google.colab import userdata, auth

# Get Groq API key from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

# Authenticate Google account
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

# Create or open FinMate spreadsheet
SHEET_NAME = "FinMate Transactions"

try:
    sh = gc.open(SHEET_NAME)
except gspread.SpreadsheetNotFound:
    sh = gc.create(SHEET_NAME)

ws = sh.sheet1

# Add headers only if the sheet is empty
if not ws.get_all_values():
    ws.append_row([
        "date",
        "transaction_type",
        "category",
        "description",
        "amount"
    ])

print("FinMate Google Sheet is ready!")
print("Sheet URL:", sh.url)

FinMate Google Sheet is ready!
Sheet URL: https://docs.google.com/spreadsheets/d/1jjgOiO7wruNhMwWOz6kyMT8cCUfYZ6aNdu00QYygvgw


In [55]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool


class Transaction(BaseModel):
    """A financial transaction for a small business."""

    date: str = Field(
        description="Date of the transaction"
    )

    transaction_type: Literal["income", "expense"] = Field(
        description="Whether the transaction is income or expense"
    )

    category: str = Field(
        description="Category such as sales, stock, electricity or rent"
    )

    description: str = Field(
        description="Short description of the transaction"
    )

    amount: float = Field(
        description="Transaction amount in rupees",
        gt=0
    )

In [6]:
@tool
def add_transaction(
    date: str,
    transaction_type: str,
    category: str,
    description: str,
    amount: float
) -> str:
    """Save a confirmed income or expense transaction to Google Sheets."""

    try:
        transaction = Transaction(
            date=date,
            transaction_type=transaction_type.lower().strip(),
            category=category.strip(),
            description=description.strip(),
            amount=amount
        )

        ws.append_row([
            transaction.date,
            transaction.transaction_type,
            transaction.category,
            transaction.description,
            transaction.amount
        ])

        return (
            f"Transaction saved successfully: "
            f"{transaction.transaction_type} of Rs {transaction.amount}."
        )

    except Exception as e:
        return f"Could not save transaction: {e}"

In [60]:
from langchain.tools import tool

@tool
def get_recent_transactions(count: int = 5) -> str:
    """Retrieve the latest saved income and expense transactions."""

    try:
        records = ws.get_all_records()

        if not records:
            return "No transactions found in Google Sheets."

        output = []

        for row in records[-max(1, min(count, 20)):][::-1]:
            data = {
                key.strip().lower().replace(" ", "_"): value
                for key, value in row.items()
            }

            output.append(
                f"Date: {data['date']} | "
                f"Amount: Rs {data['amount']} | "
                f"Category: {data['category']} | "
                f"Description: {data['description']} | "
                f"Type: {data['transaction_type']}"
            )

        return "\n".join(output)

    except Exception as e:
        return f"Error retrieving transactions: {e}"

In [7]:
@tool
def get_financial_summary() -> str:
    """Calculate total income, expenses and profit from all saved transactions."""

    try:
        records = ws.get_all_records()

        total_income = 0
        total_expenses = 0

        for row in records:
            amount = float(row["amount"] or 0)

            if row["transaction_type"].lower() == "income":
                total_income += amount

            elif row["transaction_type"].lower() == "expense":
                total_expenses += amount

        profit = total_income - total_expenses

        return (
            f"Total income: Rs {total_income:.2f}\n"
            f"Total expenses: Rs {total_expenses:.2f}\n"
            f"Profit: Rs {profit:.2f}"
        )

    except Exception as e:
        return f"Could not calculate financial summary: {e}"

In [53]:
@tool
def get_financial_summary() -> str:
    """Calculate total income, expenses and profit."""

    try:
        records = ws.get_all_records()

        total_income = 0
        total_expenses = 0

        for row in records:
            data = {
                key.strip().lower().replace(" ", "_"): value
                for key, value in row.items()
            }

            amount = float(data["amount"])

            if data["transaction_type"].lower() == "income":
                total_income += amount

            elif data["transaction_type"].lower() == "expense":
                total_expenses += amount

        profit = total_income - total_expenses

        return (
            f"Total income: Rs {total_income:.2f}\n"
            f"Total expenses: Rs {total_expenses:.2f}\n"
            f"Profit: Rs {profit:.2f}"
        )

    except Exception as e:
        return f"Error calculating summary: {e}"

In [62]:
from langchain.agents import create_agent

SYSTEM = """
ROLE:
You are FinMate AI, a friendly financial assistant
for small retail business owners.

FACTS:
You help users track income, expenses and business profit.
All financial records are stored in Google Sheets.

RULES:
1. Never invent financial values or transactions.
2. Always use get_financial_summary for financial calculations.
3. Use get_recent_transactions to retrieve saved records.
4. Before using add_transaction, collect all required details.
5. Show the transaction details and ask for confirmation
   before saving anything.
6. Save only after the user explicitly confirms.
7. Never claim a transaction was saved unless the tool succeeds.
8. If information is missing, ask the user.

FORMAT:
Keep responses simple, friendly and under 60 words in English or Telugu according to user preference.
Use Indian Rupees (Rs).
"""

agent = create_agent(
    model=llm,
    tools=[
        add_transaction,
        get_financial_summary,
        get_recent_transactions,
    ],
    system_prompt=SYSTEM
)

history = []

def chat(text: str) -> str:
    global history

    try:
        result = agent.invoke({
            "messages": history + [
                {"role": "user", "content": text}
            ]
        })

        history = result["messages"]

        return history[-1].content

    except Exception as e:
        print("Agent error:", e)
        return "Sorry, something went wrong. Please try again."

In [47]:
print(chat("Hi, I run a small grocery shop."))


Hi! 👋 How can I help you manage your grocery shop’s finances today?  
- View recent transactions  
- Add a new income or expense  
- See a profit summary  

Let me know what you’d like to do!


In [48]:
print(chat("I earned Rs 8000 from sales."))

I’m ready to record that income.  
**Transaction details**  
- Amount: Rs 8000  
- Category: Sales  
- Date: 28‑Nov‑2026  
- Description: Daily sales  
- Type: Income  

Do you want me to save this?


In [49]:
print(chat("The date is 28-11-2026. Category is sales. Save it."))


✅ Income of Rs 8000 recorded for 28‑Nov‑2026 under “Sales”.  
Anything else you’d like to add or review?


In [50]:
print(chat("I spent Rs 2000 on stock purchases today."))


✅ Expense of Rs 2000 saved for 28‑Nov‑2026 under “Stock Purchases”.  
Would you like a quick summary of your profit so far?


In [63]:
print(chat("Calculate my total business profit."))


Your business summary:  
- Total income: **Rs 8,000.00**  
- Total expenses: **Rs 2,000.00**  
- **Profit: Rs 6,000.00**  

Let me know if you’d like to see recent transactions or add a new one!


In [64]:
print(chat("Show my recent transactions."))

**Recent Transactions**

| Date | Amount | Category | Description | Type |
|------|--------|----------|-------------|------|
| 2026‑11‑28 | Rs 2,000 | Stock Purchases | Stock purchase | Expense |
| 2026‑11‑28 | Rs 8,000 | Sales | Daily sales | Income |

Let me know if you’d like to add or edit anything!
